# Gold assembly — join all sources into the base dataset

This notebook joins the cleaned silver tables (plus the MDS metadata) into one wide table,
saved as `hive_metastore.gold.gold_dataset`. It is the **assembly** step: it brings the
sources together but does **not** build the labels or the engineered features. The load
ratios, lags, rolling statistics, weather-derived features, temporal features, and the two
labels (`label_4h`, `label_24h`) are built in a later notebook from this base table.

The joins, in order:

1. **Signals + limits** — attach each transformer's current/voltage limits (`H_LIM_C`,
   `H_LIM_V`) to its readings.
2. **+ MDS metadata** — attach municipality (`CONCELHO`) and location coordinates
   (`X_SIT`, `Y_SIT`).
3. **+ Weather** — assign each transformer its nearest weather station, then attach that
   station's hourly readings.
4. **+ Event counts** — attach the 15-minute event count per transformer.
5. **Save**.

## 1. Load the sources

Load the project utilities, then read the four silver tables and the bronze MDS table.

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

In [0]:
arqlmed = spark.read.table("hive_metastore.silver.silver_arqlmed")
medidas = spark.read.table("hive_metastore.silver.silver_medidas")
mds     = spark.read.table("hive_metastore.bronze.bronze_mds")
weather = spark.read.table("hive_metastore.silver.silver_weather")
event_log = spark.read.table("hive_metastore.silver.silver_event_log")

## 2. Signals + limits

Rename the limits table's key to match, then inner-join the current and voltage limits
onto the signal readings by transformer prefix.

In [0]:
medidas = medidas.withColumnRenamed("TAG_prefix", "ID_prefix")

In [0]:
df = arqlmed.join(
    medidas.select("ID_prefix", "H_LIM_C", "H_LIM_V"),
    on="ID_prefix",
    how="inner"
)

> **Inspection.**

In [0]:
display(df)

## 3. Add MDS metadata

Attach municipality (`CONCELHO`) and coordinates (`X_SIT`, `Y_SIT`) from the MDS registry,
matching on the first 6 characters of the transformer prefix against the first 6 of
`TAGCOM`. The metadata is de-duplicated to one row per key before joining.

In [0]:
# Match on first 6 chars of ID_prefix vs first 6 chars of TAGCOM
# This is the same approach used in older_files/Connecting weather to objects.ipynb

mds_slim = mds.select(
    F.substring("TAGCOM", 1, 6).alias("mds_key"),
    "CONCELHO",
    "X_SIT",
    "Y_SIT"
    ).dropDuplicates(["mds_key"])

df = df.withColumn("mds_key", F.substring("ID_prefix", 1, 6))

df = df.join(F.broadcast(mds_slim), on="mds_key", how="inner").drop("mds_key")

In [0]:
dbutils.data.summarize(df)

> **Inspection.**

In [0]:
display(df)

## 4. Add weather

Assign each transformer the readings of its nearest weather station.

How it works:
- Snap each reading's timestamp to the hour, to match the hourly weather data.
- Take each transformer's coordinates and each station's coordinates, compute the distance
  between every transformer and every station (Haversine), and keep the nearest station
  per transformer.
- Join that station's hourly weather onto the transformer's readings.

In [0]:
# Snap DATE to the hour to match weather granularity (hourly)
df = df.withColumn("DATE_HOUR", F.date_trunc("hour", F.col("DATE")))

weather_slim = weather.select(
    F.col("DATE").alias("DATE_HOUR"),
    "location",
    "temperatura_media_do_ar_horaria_c",
    "humidade_relativa_media_horaria_percent",
    "precipitacao_horaria_mm",
    "velocidade_do_vento_media_horaria_m_per_s"
)

In [0]:
# Map each ID to its nearest weather station via X_SIT, Y_SIT
# Reuse haversine approach: get distinct ID locations and crossjoin with weather stations

id_locations = (df
    .select("ID_prefix", "X_SIT", "Y_SIT")
    .dropDuplicates(["ID_prefix"])
    .filter(F.col("X_SIT").isNotNull() & F.col("Y_SIT").isNotNull())
)

weather_stations = (weather
    .select("location", "latitude", "longitude")
    .dropDuplicates(["location"])
)

def haversine_km(lat1, lon1, lat2, lon2):
    r = F.lit(6371.0)
    dphi = F.radians(lat2 - lat1)
    dlambda = F.radians(lon2 - lon1)
    a = (
        F.pow(F.sin(dphi / 2), 2)
        + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.pow(F.sin(dlambda / 2), 2)
    )
    return r * 2 * F.asin(F.sqrt(a))

w_dist = Window.partitionBy("ID_prefix").orderBy("dist_km")

id_to_station = (
    id_locations.crossJoin(F.broadcast(weather_stations))
    .withColumn("dist_km", haversine_km(
        F.col("Y_SIT"), F.col("X_SIT"),       # lat=Y_SIT, lon=X_SIT per MDS convention
        F.col("latitude"), F.col("longitude")
    ))
    .withColumn("rn", F.row_number().over(w_dist))
    .filter(F.col("rn") == 1)
    .select("ID_prefix", "location")
)

df = df.join(F.broadcast(id_to_station), on="ID_prefix", how="inner")

df = df.join(
    weather_slim,
    on=["DATE_HOUR", "location"],
    how="inner"
).drop("DATE_HOUR", "location")

> **Inspection.**

In [0]:
display(df)

## 5. Keep the columns needed downstream

Trim to the columns the feature notebook needs. `X_SIT` / `Y_SIT` are kept for now (they
are dropped later, after they have served their purpose in the weather match).

In [0]:
df = df.select(
    "ID_prefix",
    "DATE",
    "current",
    "voltage",
    "H_LIM_C",
    "H_LIM_V",
    "CONCELHO",
    "X_SIT",
    "Y_SIT",
    "temperatura_media_do_ar_horaria_c",
    "humidade_relativa_media_horaria_percent",
    "precipitacao_horaria_mm",
    "velocidade_do_vento_media_horaria_m_per_s"
)

## 6. Add event counts

Attach the 15-minute event count per transformer.

The silver event table has one row per event, with the bucket count repeated. Here it is
first collapsed to one row per transformer-and-timestamp (`F.max`), with a check that this
leaves no duplicates, so the following left-join cannot multiply rows. Transformers with no
events get a count of 0.

In [0]:
event_log.display()

In [0]:
SHORT_ID_LEN = 6

# Step 1: Collapse event_log to ONE row per (short_id, DATE)
# Since events_15m_cnt is the same for all rows in a group, just take max/first
event_single = (
    event_log
    .select(
        F.substring("ID_prefix", 1, SHORT_ID_LEN).alias("short_id"),
        "DATE",
        "events_15m_cnt"
    )
    .groupBy("short_id", "DATE")
    .agg(F.max("events_15m_cnt").alias("evt_cnt"))
)

# Confirm: should be 0
print("Duplicates in event_single (should be 0):")
print(event_single.groupBy("short_id", "DATE").count().filter(F.col("count") > 1).count())

# Step 2: Prep main df
df = df.drop("events_15m_cnt")
df = df.withColumn("short_id", F.substring("ID_prefix", 1, SHORT_ID_LEN))

# Step 3: Left join
df = df.join(event_single, on=["short_id", "DATE"], how="left")

# Step 4: Fill nulls with 0, rename, drop helper
df = df.fillna(0, subset=["evt_cnt"])
df = df.withColumnRenamed("evt_cnt", "events_15m_cnt")
df = df.drop("short_id")

# Verify
print(f"Final row count: {df.count():,}")
df.select(
    F.sum(F.when(F.col("events_15m_cnt") > 0, 1).otherwise(0)).alias("nonzero"),
    F.mean("events_15m_cnt").alias("mean"),
    F.max("events_15m_cnt").alias("max")
).show()

> **Inspection.** Preview, spot-check one transformer, and profile the assembled table.

In [0]:
display(df)

In [0]:
display(df.filter(F.col("ID_prefix") == "ASAGD-5501-0"))

In [0]:
dbutils.data.summarize(df)

## 7. Save to gold

Create the gold schema if needed and save the assembled base table as Delta. `overwrite`
plus `overwriteSchema` makes the cell safely re-runnable.

**Target table:** `hive_metastore.gold.gold_dataset`

In [0]:
target_catalog = "hive_metastore"
target_schema  = "gold"
target_table   = "gold_dataset"

full_name = f"{target_catalog}.{target_schema}.{target_table}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

(df.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(full_name))

print(f"✅ Saved {full_name} — {df.count():,} rows")